# 08 — Model Evaluation

Comprehensive evaluation of both models: classification metrics, confusion matrix,
ROC curves, per-class performance, and inference latency measurement.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize
import joblib
import time
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname("__file__"), ".."))

## 1. Load Models and Test Data

In [ ]:
iso_model = joblib.load("../models/isolation_forest.pkl")
rf_model = joblib.load("../models/random_forest.pkl")
scaler = joblib.load("../models/scaler.pkl")
label_encoder = joblib.load("../models/label_encoder.pkl")

test_data = pd.read_csv("../data/processed/test.csv")
feature_cols = [c for c in test_data.columns if c not in ["Label", "label"]]
X_test = test_data[feature_cols].values
y_test = test_data["Label"].values

# Scale test features
X_test_scaled = scaler.transform(X_test)

print(f"Test samples: {X_test_scaled.shape[0]}")
print(f"Classes: {list(label_encoder.classes_)}")

## 2. Predictions

In [ ]:
# Anomaly detection
anomaly_scores = -iso_model.decision_function(X_test_scaled)
anomaly_preds = iso_model.predict(X_test_scaled)  # -1 = anomaly, 1 = normal

# Classification
rf_preds = rf_model.predict(X_test_scaled)
rf_proba = rf_model.predict_proba(X_test_scaled)

# Encode true labels
y_true = label_encoder.transform(y_test)
y_pred = rf_preds

print(f"Predictions generated for {len(y_pred)} samples")

## 3. Classification Metrics

In [ ]:
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("=" * 55)
print("CLASSIFICATION METRICS")
print("=" * 55)
print(f"  Accuracy:   {accuracy:.4f}")
print(f"  Precision:  {precision:.4f}")
print(f"  Recall:     {recall:.4f}")
print(f"  F1-Score:   {f1:.4f}")
print("=" * 55)

print(f"\nDetailed Classification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

## 4. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Absolute counts
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp1.plot(ax=axes[0], cmap="Blues", values_format="d")
axes[0].set_title("Confusion Matrix (Counts)")

# Normalized
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=label_encoder.classes_)
disp2.plot(ax=axes[1], cmap="Blues", values_format=".2%")
axes[1].set_title("Confusion Matrix (Normalized)")

plt.tight_layout()
plt.show()

## 5. ROC Curve

In [ ]:
# Binary ROC (BENIGN vs everything else)
benign_idx = list(label_encoder.classes_).index("BENIGN")
y_binary = (y_true != benign_idx).astype(int)
y_score_binary = 1 - rf_proba[:, benign_idx]  # probability of being attack

fpr, tpr, thresholds_roc = roc_curve(y_binary, y_score_binary)
roc_auc_val = auc(fpr, tpr)

plt.figure(figsize=(10, 7))
plt.plot(fpr, tpr, "b-", linewidth=2, label=f"ROC curve (AUC = {roc_auc_val:.4f})")
plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — BENIGN vs Attack")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Per-Class Metrics

In [ ]:
n_classes = len(label_encoder.classes_)
y_true_bin = label_binarize(y_true, classes=range(n_classes))

per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0)
per_class_recall = recall_score(y_true, y_pred, average=None, zero_division=0)
per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

x_pos = np.arange(n_classes)
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x_pos - width, per_class_precision, width, label="Precision", alpha=0.85)
ax.bar(x_pos, per_class_recall, width, label="Recall", alpha=0.85)
ax.bar(x_pos + width, per_class_f1, width, label="F1-Score", alpha=0.85)

ax.set_xticks(x_pos)
ax.set_xticklabels(label_encoder.classes_, rotation=45, ha="right")
ax.set_ylabel("Score")
ax.set_title("Per-Class Metrics")
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Detection Latency

In [ ]:
# Measure inference time per sample
n_runs = 1000
sample_idx = np.random.choice(len(X_test_scaled), size=min(n_runs, len(X_test_scaled)), replace=False)
X_sample = X_test_scaled[sample_idx]

# Warm up
for _ in range(10):
    iso_model.decision_function(X_sample[:1])
    rf_model.predict(X_sample[:1])

# Time anomaly detection
iso_times = []
for i in range(len(X_sample)):
    t0 = time.perf_counter()
    iso_model.decision_function(X_sample[i:i+1])
    iso_times.append(time.perf_counter() - t0)

# Time classification
rf_times = []
for i in range(len(X_sample)):
    t0 = time.perf_counter()
    rf_model.predict(X_sample[i:i+1])
    rf_times.append(time.perf_counter() - t0)

iso_times = np.array(iso_times) * 1000  # ms
rf_times = np.array(rf_times) * 1000  # ms
total_times = iso_times + rf_times

print("=" * 55)
print("DETECTION LATENCY (per sample)")
print("=" * 55)
print(f"  Isolation Forest:")
print(f"    Mean:   {iso_times.mean():.3f} ms")
print(f"    Std:    {iso_times.std():.3f} ms")
print(f"    P95:    {np.percentile(iso_times, 95):.3f} ms")
print(f"    P99:    {np.percentile(iso_times, 99):.3f} ms")
print(f"  Random Forest:")
print(f"    Mean:   {rf_times.mean():.3f} ms")
print(f"    Std:    {rf_times.std():.3f} ms")
print(f"    P95:    {np.percentile(rf_times, 95):.3f} ms")
print(f"    P99:    {np.percentile(rf_times, 99):.3f} ms")
print(f"  Combined:")
print(f"    Mean:   {total_times.mean():.3f} ms")
print(f"    P95:    {np.percentile(total_times, 95):.3f} ms")
print(f"    P99:    {np.percentile(total_times, 99):.3f} ms")
print("=" * 55)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(iso_times, bins=50, alpha=0.7, label="Isolation Forest")
axes[0].hist(rf_times, bins=50, alpha=0.7, label="Random Forest")
axes[0].set_xlabel("Latency (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title("Inference Latency Distribution")
axes[0].legend()

axes[1].boxplot([iso_times, rf_times], labels=["IsoForest", "RandForest"])
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Latency Box Plot")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Summary Report

In [ ]:
print("\n" + "=" * 60)
print("COMPREHENSIVE EVALUATION SUMMARY")
print("=" * 60)
print(f"
  Dataset:
    Test samples:      {len(X_test_scaled)}
    Features:          {X_test_scaled.shape[1]}
    Classes:           {n_classes}
    Class names:       {list(label_encoder.classes_)}")
print(f"
  Classification Performance:
    Accuracy:          {accuracy:.4f}
    Weighted Precision:{precision:.4f}
    Weighted Recall:   {recall:.4f}
    Weighted F1:       {f1:.4f}
    ROC AUC (binary):  {roc_auc_val:.4f}")
print(f"
  Latency (per sample):
    Combined Mean:     {total_times.mean():.3f} ms
    Combined P99:      {np.percentile(total_times, 99):.3f} ms
    Throughput (est):  ~{1000/total_times.mean():.0f} samples/sec")
print(f"
  Anomaly Detection:
    Total anomalies:   {sum(anomaly_preds == -1)}
    Anomaly rate:      {sum(anomaly_preds == -1)/len(anomaly_preds)*100:.1f}%")
print("\n" + "=" * 60)